In [13]:
# import pytorch
import torch

import os
import pandas as pd
from torchvision.io import read_image
from torch.utils.data import Dataset
from torchvision.transforms import ToTensor, Lambda

import source.preprocessing as cheepre

In [11]:
import importlib; importlib.reload(cheepre)

<module 'source.preprocessing' from '/Users/m.wehrens/Documents/git_repos/_UVA/2025_Cheeky-cells/source/preprocessing.py'>

In [ ]:
# Based on tutorial and notebook listed below
# https://pytorch.org/tutorials/beginner/basics/intro.html
# /Users/m.wehrens/Documents/git_repos/_UVA/2025_MW-testing-ML/simple-tutorial.ipynb



class CustomImageDataset(Dataset):
    def __init__(self, annot_dir, img_dir, transform=None, target_transform=None):
        
        self.annot_dir = annot_dir
        self.img_dir = img_dir
        
        # get dataset info
        annot_pixelcount_list, list_allimgpaths, list_annotfilepaths = cheepre.acquire_trainingset_info(img_dir, annot_dir)
        self.annot_pixelcount_list = annot_pixelcount_list
        self.img_list = list_allimgpaths
        self.img_annot_list = list_annotfilepaths
        
        # get dataset img_idx, locx, locy, label
        self.img_idxs, self.posis, self.posjs, self.labels = \
            cheepre.build_labels_and_positions(annot_dir, annot_pixelcount_list, list_allimgpaths, list_annotfilepaths)
        self.labels = self.labels.astype(int)
        
        # transforms        
        self.transform = transform
        self.target_transform = target_transform

    def __len__(self):
        return len(self.img_labels)

    def __getitem__(self, idx):
        
        # now produce the label and the image
        label = self.labels[idx]
        image = cheepre.provide_crop(self.img_dir, self.img_list[idx], self.posis[idx], self.posjs[idx])
        
        # print(image)
        print(type(label))
    
        # transform
        if self.transform:
            image = self.transform(image)
            
        if self.target_transform:
            label = self.target_transform(label)
            
        return image, label

In [47]:
%%capture
ANNOT_DIR = '/Users/m.wehrens/Data_UVA/2024_07_Wang-cel/2025_Cells_preliminarybatch1/Cheeck-Cells_AnnotatedMW_resized_humanannotated/'
IMG_DIR   = '/Users/m.wehrens/Data_UVA/2024_07_Wang-cel/2025_Cells_preliminarybatch1/Cheeck-Cells_AnnotatedMW_resized_grey/'
mydataset = CustomImageDataset(annot_dir=ANNOT_DIR, 
                               img_dir=IMG_DIR, 
                               transform=ToTensor(), 
                               target_transform=Lambda(lambda y: torch.zeros(6, dtype=torch.float).scatter_(0, torch.tensor(y), value=1)))

In [48]:


# show one of the images of "mydataset" above
#import matplotlib.pyplot as plt
#import numpy as np
#import random
#import torchvision.transforms.functional as TF

image, label = mydataset[0]


<class 'numpy.uint64'>


TypeError: an integer is required